In [1]:
import zipfile
import pandas as pd
import re
import nltk
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
import umap
import hdbscan
from scipy.stats import chi2_contingency


/home/onyxia/work/NLP_3A/nlp-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Analyse rapide des données brutes

Pour l'instant, on a uniquement les fichiers txt de l'élection de 1981 mais on a les métadonnées pour tout
Il faudra adapter pour intégrer toutes les années

In [2]:
#lecture des métadonnées
metadonnees = pd.read_csv("data/archelect_search.zip", compression="zip")

/tmp/ipykernel_24969/3025274146.py:2: DtypeWarning: Columns (0: departement-nom, 1: departement-insee, 2: identifiant de circonscription, 3: pdf, 4: suppleant-nom, 5: suppleant-prenom, 6: suppleant-sexe, 7: suppleant-age, 8: suppleant-age-calcule, 9: suppleant-age-tranche, 10: suppleant-profession, 11: suppleant-mandat-en-cours, 12: suppleant-mandat-passe, 13: suppleant-associations, 14: suppleant-autres-statuts, 15: suppleant-soutien, 16: suppleant-liste, 17: suppleant-decorations) have mixed types. Specify dtype option on import or set low_memory=False.
  metadonnees = pd.read_csv("data/archelect_search.zip", compression="zip")


In [3]:
zip_path = "data/legislatives.zip"

all_texts = []

with zipfile.ZipFile(zip_path, "r") as z:
    
    # on liste les fichiers txt
    txt_files = [f for f in z.namelist() if f.endswith(".txt")]
    
    # on filtre les dossiers parasites type __MACOSX
    txt_filtres = [f for f in txt_files if not f.startswith("__MACOSX")]
    
    print(f"Nombre de fichiers txt valides : {len(txt_filtres)}")

    for file in txt_filtres:
        try:
            with z.open(file) as f:
                text_content = f.read().decode("utf-8")
                # garder uniquement le nom du fichier (sans dossier)
                filename = file.split("/")[-1].replace(".txt", "")
                #print(filename)
                
                all_texts.append({
                    "id": filename,
                    "text": text_content
                })
        
        except Exception as e:
            print(f"Erreur lors de la lecture de {file} : {e}")

# --- DataFrame textes ---
all_texts_df = pd.DataFrame(all_texts)

# --- Fusion texte + métadonnées ---
meta_et_texts = metadonnees.merge(all_texts_df, on="id", how="right")

# --- Vérification des correspondances ---
missing_meta = meta_et_texts[meta_et_texts["text"].isnull()]["id"]

if len(missing_meta) > 0:
    print("Certains fichiers n'ont pas de texte associé :")
    print(missing_meta.tolist())

print(meta_et_texts.head())
print(f"Nombre total de documents : {len(meta_et_texts)}")


Nombre de fichiers txt valides : 3182
                               id        date  \
0  EL136_L_1981_06_075_12_1_PF_03  1981-06-14   
1  EL134_L_1981_06_035_01_1_PF_01  1981-06-14   
2  EL135_L_1981_06_057_06_1_PF_04  1981-06-14   
3  EL136_L_1981_06_067_07_1_PF_02  1981-06-14   
4  EL134_L_1981_06_013_06_1_PF_02  1981-06-14   

                                             subject  \
0  Ve République;France;Élections législatives;As...   
1  France;Assemblée Nationale;Élections législati...   
2  Assemblée Nationale;France;Élections législati...   
3  France;Ve République;Assemblée Nationale;Élect...   
4  Élections législatives;Assemblée Nationale;Ve ...   

                                               title contexte-election  \
0  Élections législatives de 1981, Paris - 75, ci...      législatives   
1  Élections législatives de 1981, Ille-et-Vilain...      législatives   
2  Élections législatives de 1981, Moselle - 57, ...      législatives   
3  Élections législatives de 1981,

In [ ]:
zip_path = "data/legislatives.zip"

all_texts = []

with zipfile.ZipFile(zip_path, "r") as z:
    
    # on liste les fichiers txt
    txt_files = [f for f in z.namelist() if f.endswith(".txt")]
    
    # on filtre les dossiers parasites type __MACOSX
    txt_filtres = [f for f in txt_files if not f.startswith("__MACOSX")]
    
    print(f"Nombre de fichiers txt valides : {len(txt_filtres)}")

    for file in txt_filtres:
        try:
            with z.open(file) as f:
                text_content = f.read().decode("utf-8")
                # garder uniquement le nom du fichier (sans dossier)
                filename = file.split("/")[-1].replace(".txt", "")
                #print(filename)
                
                all_texts.append({
                    "id": filename,
                    "text": text_content
                })
        
        except Exception as e:
            print(f"Erreur lors de la lecture de {file} : {e}")

# --- DataFrame textes ---
all_texts_df = pd.DataFrame(all_texts)

# --- Fusion texte + métadonnées ---
meta_et_texts = metadonnees.merge(all_texts_df, on="id", how="right")

# --- Vérification des correspondances ---
missing_meta = meta_et_texts[meta_et_texts["text"].isnull()]["id"]

if len(missing_meta) > 0:
    print("Certains fichiers n'ont pas de texte associé :")
    print(missing_meta.tolist())

print(meta_et_texts.head())
print(f"Nombre total de documents : {len(meta_et_texts)}")


Nombre de fichiers txt valides : 3182
                               id        date  \
0  EL136_L_1981_06_075_12_1_PF_03  1981-06-14   
1  EL134_L_1981_06_035_01_1_PF_01  1981-06-14   
2  EL135_L_1981_06_057_06_1_PF_04  1981-06-14   
3  EL136_L_1981_06_067_07_1_PF_02  1981-06-14   
4  EL134_L_1981_06_013_06_1_PF_02  1981-06-14   

                                             subject  \
0  Ve République;France;Élections législatives;As...   
1  France;Assemblée Nationale;Élections législati...   
2  Assemblée Nationale;France;Élections législati...   
3  France;Ve République;Assemblée Nationale;Élect...   
4  Élections législatives;Assemblée Nationale;Ve ...   

                                               title contexte-election  \
0  Élections législatives de 1981, Paris - 75, ci...      législatives   
1  Élections législatives de 1981, Ille-et-Vilain...      législatives   
2  Élections législatives de 1981, Moselle - 57, ...      législatives   
3  Élections législatives de 1981,

In [4]:
meta_et_texts.head()

,id,date,subject,title,contexte-election,contexte-tour,cote,departement,departement-nom,departement-insee,...,suppleant-age-tranche,suppleant-profession,suppleant-mandat-en-cours,suppleant-mandat-passe,suppleant-associations,suppleant-autres-statuts,suppleant-soutien,suppleant-liste,suppleant-decorations,text
0,EL136_L_1981_06_075_12_1_PF_03,1981-06-14,Ve République;France;Élections législatives;As...,"Élections législatives de 1981, Paris - 75, ci...",législatives,1.0,EL136,75,Paris,75 - Paris (Seine),...,entre 30 et 39 ans,employé PTT,non mentionné,non mentionné,non mentionné,non mentionné,Lutte ouvrière,non mentionné,non,Sciences Po / fonds CEVIPOF\nElections législa...
1,EL134_L_1981_06_035_01_1_PF_01,1981-06-14,France;Assemblée Nationale;Élections législati...,"Élections législatives de 1981, Ille-et-Vilain...",législatives,1.0,EL134,35,Ille-et-Vilaine,35 - Ille-et-Vilaine,...,entre 20 et 29 ans,institutrice,non mentionné,non mentionné,non mentionné,non mentionné,Lutte ouvrière,non mentionné,non,Sciences Po / fonds CEVIPOF\nElections législa...
2,EL135_L_1981_06_057_06_1_PF_04,1981-06-14,Assemblée Nationale;France;Élections législati...,"Élections législatives de 1981, Moselle - 57, ...",législatives,1.0,EL135,57,Moselle,57 - Moselle,...,entre 20 et 29 ans,surveillant internat,non mentionné,non mentionné,non mentionné,non mentionné,Lutte ouvrière,non mentionné,non,Élections législatives de juin 1981 6e circons...
3,EL136_L_1981_06_067_07_1_PF_02,1981-06-14,France;Ve République;Assemblée Nationale;Élect...,"Élections législatives de 1981, Bas-Rhin - 67,...",législatives,1.0,EL136,67,Bas-Rhin,67 - Bas-Rhin,...,non mentionné,ouvrier spécialisé,non mentionné,non mentionné,non mentionné,non mentionné,Parti socialiste,non mentionné,non,Sciences Po / fonds CEVIPOF\na\nELECTIONS LEGI...
4,EL134_L_1981_06_013_06_1_PF_02,1981-06-14,Élections législatives;Assemblée Nationale;Ve ...,"Élections législatives de 1981, Bouches-du-Rhô...",législatives,1.0,EL134,13,Bouches-du-Rhône,13 - Bouches-du-Rhône,...,non mentionné,employée,non mentionné,non mentionné,non mentionné,non mentionné,Parti communiste français,Union de la gauche,non,RÉPUBLIQUE FRANÇAISE · LIBERTÉ ÉGALITÉ FRATERN...


## Pre-traitement des données

In [5]:
nltk.download('stopwords')
stop_words = set(stopwords.words('french'))


[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [7]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    words = [w for w in text.split() if w not in stop_words]
    return ' '.join(words)

meta_et_texts['texte_clean'] = meta_et_texts['text'].apply(clean_text)
meta_et_texts.drop_duplicates(inplace=True)
print(f"Dataset chargé : {meta_et_texts.shape[0]} lignes")

Dataset chargé : 3182 lignes


## Représentation textuelle

In [9]:
model = SentenceTransformer('camembert-base')
embeddings = model.encode(meta_et_texts['texte_clean'].tolist(), batch_size=16, show_progress_bar=True)
print(f"Embeddings : {embeddings.shape}")

No sentence-transformers model found with name camembert-base. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 736.10it/s, Materializing param=pooler.dense.weight]                               
CamembertModel LOAD REPORT from: camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 199/199 [13:14<00:00,  3.99s/it]

Embeddings : (3182, 768)


## Classification par thème

In [10]:
umap_embeddings = umap.UMAP(
    n_neighbors=15, min_dist=0.0, n_components=5, random_state=42
).fit_transform(embeddings)
print(f"UMAP embeddings shape: {umap_embeddings.shape}")

/opt/python/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP embeddings shape: (3182, 5)


In [11]:
print("Clustering avec HDBSCAN...")
clusterer = hdbscan.HDBSCAN(min_cluster_size=10)
meta_et_texts['theme_cluster'] = clusterer.fit_predict(umap_embeddings)
print(f"Clusters trouvés : {meta_et_texts['theme_cluster'].nunique()} (le -1 correspond aux outliers)")


Clustering avec HDBSCAN...
Clusters trouvés : 17 (le -1 correspond aux outliers)


## Croisement avec les professions des candidats

On fait un peu de visualisation

In [12]:
plt.figure(figsize=(12,6))
sns.countplot(x='theme_cluster', hue='profession', data=meta_et_texts)
plt.title("Répartition des thèmes abordés par métier")
plt.xlabel("Thème")
plt.ylabel("Nombre de professions de foi")
plt.legend(title="Métier")
plt.show()

ValueError: Could not interpret value `profession` for `hue`. An entry with this name does not appear in `data`.

<Figure size 1200x600 with 0 Axes>

## Analyses statistiques diverses

In [ ]:
contingency_table = pd.crosstab(meta_et_texts['profession'], meta_et_texts['theme_cluster'])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi2 = {chi2:.2f}, p-value = {p:.4f}")